In [ ]:
# Cell 1: Import necessary LangChain and local LLM modules
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
from langchain_classic.chains import RetrievalQA

print("Imports successful!")

# Cell 2: Initialize embeddings and load the existing Chroma DB from disk
# CRITICAL: This must be the exact same model used to create the DB!
embedding_model = OllamaEmbeddings(model="mistral:7b")

# Load the database without re-ingesting
db = Chroma(
    persist_directory="./chroma_vector_db", 
    embedding_function=embedding_model
)

print("Database loaded successfully from disk.")

# Cell 3: Verify the contents of the database
collection_count = db._collection.count()

print(f"Successfully connected to Chroma DB.")
print(f"Total document chunks ready for querying: {collection_count}")

In [ ]:
# Cell 4: Initialize the LLM that will generate the final answer
llm = ChatOllama(model="mistral:7b", temperature=0) # temperature=0 keeps answers factual

print("LLM Initialized.")

In [ ]:
# Cell 5: Rebuild the chain to expose the sources
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True  # <--- THIS IS THE MAGIC KEY
)

print("Chain updated to return sources!")

# Cell 6: Ask a question relevant to the documents you ingested
question = "How does the auth system store passwords, and how is that similar to MySQL's user management?" # Change this to a real question

# Run the chain
print("Thinking...\n")
response = qa_chain.invoke({"query": question})

print(f"Question: {question}\n")
print(f"Answer: {response['result']}")

print(f"Answer: {response['result']}\n")
print("-" * 40)
print("SOURCES USED:\n")

# Loop through the source documents and print their metadata and a snippet of text
for i, doc in enumerate(response['source_documents']):
    print(f"Source {i+1}:")
    print(f"Metadata: {doc.metadata}") # This holds your page number, source file, etc.
    print(f"Text Snippet: {doc.page_content[:150]}...\n") # Print the first 150 characters